In [1]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import nibabel as nib
from scipy.interpolate import interpn
from tqdm.notebook import tqdm
import pandas as pd


In [2]:
def apply_affine(tx,
                 ty,
                 tz,
                 rx,
                 ry,
                 rz,
                 image3d,
                 resolution=(1, 1, 1),
                 method='linear',
                 fill_value=0):

    # Define the affine transformation matrix

    # Create the rotation matrix
    rotx = np.array([[1, 0, 0], [0, np.cos(rx), -np.sin(rx)],
                     [0, np.sin(rx), np.cos(rx)]])

    roty = np.array([[np.cos(ry), 0, np.sin(ry)], [0, 1, 0],
                     [-np.sin(ry), 0, np.cos(ry)]])

    rotz = np.array([[np.cos(rz), -np.sin(rz), 0], [np.sin(rz),
                                                    np.cos(rz), 0], [0, 0, 1]])

    rotation_matrix = np.dot(np.dot(rotx, roty), rotz)

    # Create the affine transformation matrix
    affine = np.array([[
        rotation_matrix[0, 0], rotation_matrix[0, 1], rotation_matrix[0, 2], tx
    ], [
        rotation_matrix[1, 0], rotation_matrix[1, 1], rotation_matrix[1, 2], ty
    ], [
        rotation_matrix[2, 0], rotation_matrix[2, 1], rotation_matrix[2, 2], tz
    ], [0, 0, 0, 1]])

    # create grids of the original image
    center = np.array(image3d.shape) / 2.0

    grids = []
    for dim in range(3):
        grid = np.arange(image3d.shape[dim]) - center[dim]
        grid *= resolution[dim]
        grids.append(grid)

    # Create a meshgrid of coordinates
    x, y, z = np.meshgrid(grids[0], grids[1], grids[2], indexing='ij')

    # Create a coordinates array
    coords = np.array(
        [x.flatten(),
         y.flatten(),
         z.flatten(),
         np.ones_like(x.flatten())]).T

    # Apply the affine transformation
    transformed_coords = np.dot(affine, coords.T).T

    # Interpolate the image values
    image3d2 = interpn(grids,
                       image3d,
                       transformed_coords[:, :3],
                       method=method,
                       bounds_error=False,
                       fill_value=fill_value)
    # Reshape the rotated image
    image3d2 = image3d2.reshape(image3d.shape)

    return image3d2


In [3]:
def rotate_nii(rot_x,
               rot_y,
               rot_z,
               image_path,
               save_path,
               method='linear',
               fill_value=0):
    angle_x = rot_x * np.pi / 180.0
    angle_y = rot_y * np.pi / 180.0
    angle_z = rot_z * np.pi / 180.0

    nii = nib.load(image_path)
    resolution = nii.header.get_zooms()
    resolution = (resolution[1], resolution[0], resolution[2])
    image = nii.get_fdata()

    image = apply_affine(0,
                         0,
                         0,
                         angle_x,
                         angle_y,
                         angle_z,
                         image,
                         resolution=resolution,
                         method=method,
                         fill_value=fill_value)

    new_nii = nib.Nifti1Image(image, nii.affine, nii.header)

    # Save the new image to a file
    nib.save(new_nii, save_path)


# rotate_nii(0, 0, 30, image_path='debug/049.nii', save_path='debug/049_aug1.nii')
# rotate_nii(10, 0, 0, image_path='debug/049.nii', save_path='debug/049_aug2.nii')
# rotate_nii(0, 10, 0, image_path='debug/049.nii', save_path='debug/049_aug3.nii')

# rotate_nii(0,
#            0,
#            30,
#            image_path='debug/049_mask.nii',
#            save_path='debug/049_mask_aug1.nii',
#            method='nearest')
# rotate_nii(10,
#            0,
#            0,
#            image_path='debug/049_mask.nii',
#            save_path='debug/049_mask_aug2.nii',
#            method='nearest')
# rotate_nii(0,
#            10,
#            0,
#            image_path='debug/049_mask.nii',
#            save_path='debug/049_mask_aug3.nii',
#            method='nearest')


In [4]:
image_path = '/dataNAS/datasets/physionet-ich/ct-ich/1.3.1/ct_scans'
label_path = '/dataNAS/datasets/physionet-ich/ct-ich/1.3.1/masks'

save_path = '/home/zhongnan/Datasets/physionet-ich-sim/rotz30_rand_nii'

rotz_history = []

for root, _, files in os.walk(image_path):
    pbar = tqdm(files)

    for file in pbar:

        pbar.set_description(file)

        if not file.endswith('nii'):
            continue

        if np.random.randint(2) == 0:
            rotz = 30 + 2 * np.random.randn()
        else:
            rotz = -30 + 2 * np.random.randn()

        rotz_history.append({'file': file, 'rotz': rotz})

        rotate_nii(0,
                   0,
                   rotz,
                   image_path=os.path.join(image_path, file),
                   save_path=os.path.join(save_path, 'images', file))
        rotate_nii(0,
                   0,
                   rotz,
                   image_path=os.path.join(label_path, file),
                   save_path=os.path.join(save_path, 'masks', file),
                   method='nearest')

df = pd.DataFrame(rotz_history)
df.to_excel(os.path.join(save_path, 'rotation.xlsx'), index=False)

  0%|          | 0/76 [00:00<?, ?it/s]